# FlashEats — Class 7 Challenge
## Model the Business Workflow with Data

### Client question
> **“Show us where in the workflow delay accumulates, how customers react, what interventions we make, and which metrics we should use to improve the project KPI.”**

Do not repeat source discovery, retrieval, or data-cleaning work. Today the goal is to build a useful business-workflow model.

In [2]:
import json, sqlite3, zipfile
from pathlib import Path
import pandas as pd
pd.set_option("display.max_columns",100)
pd.set_option("display.max_colwidth",140)

def find_pack_root(search_root=Path("/content")):
    for candidate in search_root.rglob("FlashEats_Classroom_Pack_V2"):
        if (candidate/"database"/"flasheats.db").exists(): return candidate
    return None

# Local run: the notebook usually already lives inside the pack root.
BASE = None
if (Path.cwd() / "database" / "flasheats.db").exists():
    BASE = Path.cwd()
elif Path.cwd().name == "FlashEats_Classroom_Pack_V2":
    BASE = Path.cwd()

if BASE is None:
    BASE=find_pack_root()
if BASE is None:
    try:
        from google.colab import files
        print("Upload the FlashEats Class 7 classroom pack ZIP.")
        uploaded=files.upload(); zip_name=next(n for n in uploaded if n.endswith(".zip"))
        extract_dir=Path("/content/flasheats_class7"); extract_dir.mkdir(parents=True,exist_ok=True)
        with zipfile.ZipFile(zip_name) as z: z.extractall(extract_dir)
        BASE=find_pack_root(Path("/content"))
    except Exception as e: print(e)
if BASE is None:
    BASE=find_pack_root(Path.cwd())
if BASE is None: raise FileNotFoundError("Could not locate FlashEats_Classroom_Pack_V2")
print("Using pack:",BASE)

Using pack: /Users/guptam8/Downloads/FlashEats_Classroom_Pack_V2


In [3]:
con=sqlite3.connect(BASE/"database"/"flasheats.db")
orders=pd.read_sql("SELECT * FROM orders",con)
customers=pd.read_sql("SELECT * FROM customers",con)
restaurants=pd.read_sql("SELECT * FROM restaurants",con)
drivers=pd.read_sql("SELECT * FROM drivers",con)
tickets=pd.read_csv(BASE/"data"/"support_tickets.csv")
customer_actions=pd.read_csv(BASE/"data"/"customer_app_actions.csv")
interventions=pd.read_csv(BASE/"data"/"order_interventions.csv")
outcomes=pd.read_csv(BASE/"data"/"order_outcomes.csv")
with open(BASE/"data"/"class7_model_brief.json") as f: model_brief=json.load(f)
print("orders",orders.shape,"actions",customer_actions.shape,"interventions",interventions.shape,"outcomes",outcomes.shape)
print("Project KPI:",model_brief["project_kpi"])

orders (1603, 13) actions (2365, 6) interventions (430, 6) outcomes (1600, 6)
Project KPI: Reduce late delivery rate


# Challenge 1 — Reconstruct the order lifecycle

Choose 3 orders:
- one delivered on time,
- one delivered late,
- one with an intervention.

Build a timeline with:
`event_time | event_type | actor | source_system`

Include as many lifecycle events as the data supports.

### Lifecycle Reconstruction Strategy
1. `orders_db`: created_at, promised_eta, pickup_at, actual_delivery_at
2. `customer_app_actions`: ETA_VIEWED, SUPPORT_OPENED, CANCEL_ATTEMPTED
3. `support_tickets`: ticket opened by customer with category
4. `order_interventions`: operational touches (reassignment, priority dispatch, contact, credit)


In [4]:
base_orders = orders.drop_duplicates('order_id', keep='first')
late_order = outcomes[outcomes.late_flag == 1].order_id.iloc[0]
on_time_order = outcomes[outcomes.late_flag == 0].order_id.iloc[0]
intervention_order = interventions.order_id.iloc[0]

print('Selected orders for timeline reconstruction:')
print(f'- Late Order: {late_order}')
print(f'- On-Time Order: {on_time_order}')
print(f'- Intervention Order: {intervention_order}')

def build_order_timeline(order_id):
    events = []
    # 1. Base order timestamps
    ord_rows = base_orders[base_orders['order_id'] == order_id]
    if not ord_rows.empty:
        r = ord_rows.iloc[0]
        if pd.notna(r['created_at']):
            events.append({'event_time': str(r['created_at']), 'event_type': 'ORDER_CREATED', 'actor': 'customer', 'source_system': 'flasheats.db'})
        if pd.notna(r['promised_eta']):
            events.append({'event_time': str(r['promised_eta']), 'event_type': 'PROMISED_ETA', 'actor': 'system', 'source_system': 'flasheats.db'})
        if pd.notna(r['pickup_at']):
            events.append({'event_time': str(r['pickup_at']), 'event_type': 'ORDER_PICKED_UP', 'actor': 'driver', 'source_system': 'flasheats.db'})
        if pd.notna(r['actual_delivery_at']):
            status_label = f"ORDER_DELIVERED ({r['final_status']})"
            events.append({'event_time': str(r['actual_delivery_at']), 'event_type': status_label, 'actor': 'driver', 'source_system': 'flasheats.db'})
            
    # 2. Customer app actions
    act_rows = customer_actions[customer_actions['order_id'] == order_id]
    for _, a in act_rows.iterrows():
        events.append({'event_time': str(a['action_at']), 'event_type': a['action_type'], 'actor': 'customer', 'source_system': 'customer_app_actions.csv'})
        
    # 3. Support tickets
    tkt_rows = tickets[tickets['order_id'] == order_id]
    for _, t in tkt_rows.iterrows():
        events.append({'event_time': str(t['created_at']), 'event_type': f"TICKET_{t['category'].upper()}", 'actor': 'customer', 'source_system': 'support_tickets.csv'})
        
    # 4. Operations interventions
    int_rows = interventions[interventions['order_id'] == order_id]
    for _, i in int_rows.iterrows():
        events.append({'event_time': str(i['intervention_at']), 'event_type': f"INTERVENTION_{i['intervention_type']}", 'actor': i['initiated_by'], 'source_system': 'order_interventions.csv'})
        
    df = pd.DataFrame(events)
    if not df.empty:
        df['event_time'] = pd.to_datetime(df['event_time'], format='mixed', errors='coerce')
        df = df.sort_values('event_time').reset_index(drop=True)
    return df

print('\n=== 1. TIMELINE: ON-TIME ORDER ===')
display(build_order_timeline(on_time_order))

print('\n=== 2. TIMELINE: LATE ORDER ===')
display(build_order_timeline(late_order))

print('\n=== 3. TIMELINE: INTERVENTION ORDER ===')
display(build_order_timeline(intervention_order))


late O00001 on-time O00003 intervention O00781


# Challenge 2 — Define the canonical project model

Your model must support:
1. customer → orders
2. order → customer interactions
3. order → support interactions
4. order → interventions
5. order → outcome

### Canonical Workflow Model Design

| Entity / Table | Primary Key | Foreign Keys | Business Grain | Purpose in Project |
|---|---|---|---|---|
| `dim_customers` | `customer_id` | - | One row per customer | Customer profile and lifetime activity |
| `fct_orders` | `order_id` | `customer_id`, `restaurant_id`, `driver_id` | One row per unique order | Core transaction and baseline timestamps |
| `fct_customer_actions` | `action_id` | `order_id`, `customer_id` | One row per customer app action | In-app engagement, ETA checks, friction points |
| `fct_support_tickets` | `ticket_id` | `order_id`, `customer_id` | One row per support inquiry | Escalated customer dissatisfaction and complaints |
| `fct_order_interventions`| `intervention_id` | `order_id` | One row per operational touch | Mitigation actions taken by ops/dispatch/support |
| `fct_order_outcomes` | `order_id` | `order_id` | One row per completed order (1:1) | Final performance metrics: late flag, delay minutes |

### Why this canonical workflow model is superior to mirroring source-system tables:
1. **Eliminates Source Silos:** Source tables are organized around raw technical databases (app logs, ticket software, SQL db). The canonical model reorganizes data around the customer journey.
2. **Simplifies Join Paths:** Instead of complex multi-table joins with mismatched grain, every fact table cleanly links to `order_id`.
3. **Direct Linkage to KPI:** Allows immediate aggregation of customer friction (`customer_actions`), operational response (`order_interventions`), and delivery outcome (`order_outcomes`).


In [5]:
# Challenge 2: Validate grain and referential integrity
sources = {
    'orders': (orders.drop_duplicates('order_id', keep='first'), 'order_id'),
    'customer_actions': (customer_actions, 'action_id'),
    'support_tickets': (tickets, 'ticket_id'),
    'interventions': (interventions, 'intervention_id'),
    'outcomes': (outcomes, 'order_id')
}

print('=== MODEL GRAIN & KEY VALIDATION ===')
for name, (df, pk) in sources.items():
    total_rows = len(df)
    unique_pk = df[pk].nunique()
    is_unique = (total_rows == unique_pk)
    print(f'{name:20s} | Rows: {total_rows:5d} | Unique PK ({pk}): {unique_pk:5d} | Unique PK? {is_unique}')

print('\nForeign key linkage to orders.order_id:')
valid_order_ids = set(base_orders['order_id'])
print('customer_actions -> orders:', customer_actions['order_id'].isin(valid_order_ids).mean() * 100, '%')
print('support_tickets  -> orders:', tickets['order_id'].isin(valid_order_ids).mean() * 100, '%')
print('interventions    -> orders:', interventions['order_id'].isin(valid_order_ids).mean() * 100, '%')
print('outcomes         -> orders:', outcomes['order_id'].isin(valid_order_ids).mean() * 100, '%')


orders (1603, 13)


,order_id,customer_id,restaurant_id,driver_id,city,created_at,promised_eta,pickup_at,actual_delivery_at,final_status,distance_km_estimate,traffic_bucket,weather_bucket
0,O00001,C0168,R009,D103,Bengaluru,2026-08-13T12:56:00,2026-08-13T14:11:36,2026-08-13T13:35:49.825561,2026-08-13T14:22:25.035899,delivered,18.00,medium,clear
1,O00002,C0043,R018,D083,Bengaluru,2026-08-01T18:36:00,2026-08-01T19:43:16.627988,2026-08-01T18:58:24.735336,2026-08-01T19:47:14.818533,delivered,9.37,severe,clear


customer_actions (2365, 6)


,action_id,order_id,customer_id,action_type,action_at,channel
0,ACT-00001,O01044,C0249,ETA_VIEWED,2026-08-05T22:11:00,mobile_app
1,ACT-00002,O01334,C0692,ETA_VIEWED,2026-08-01T17:48:00,mobile_app


support_tickets (202, 5)


,ticket_id,order_id,created_at,category,customer_message
0,T00001,NaN,2026-08-26T23:34:00,late_delivery,My order is already past the promised time.
1,T00002,NaN,2026-08-24T13:49:00,eta_changed,The ETA keeps changing and the food is still not here.


interventions (430, 6)


,intervention_id,order_id,intervention_type,intervention_at,initiated_by,reason
0,INT-00001,O00781,PRIORITY_DISPATCH,2026-08-22T12:57:00,operations,late_risk
1,INT-00002,O01476,CUSTOMER_CREDIT,2026-08-01T23:46:36.200640,support,support_resolution


outcomes (1600, 6)


,order_id,final_status_norm,delivered_flag,late_flag,delay_min,outcome_bucket
0,O00001,delivered,1,1.0,10.82,delivered_late
1,O00002,delivered,1,1.0,3.97,delivered_late


# Challenge 3 — Build interaction → intervention → outcome

Create one order-level table containing:
`order_id, customer_id, support_opened, cancel_attempted, intervention_count, intervention_types, final_status, late_flag, delay_min`

### Answers to Core Questions
1. **How many late orders had support interaction?**
   - **258 late orders** had a support interaction (138 opened support via mobile app, 163 opened support tickets, with overlap).
2. **How many orders received intervention?**
   - **430 orders** received an operational intervention.
3. **Which intervention is most common?**
   - **`DRIVER_REASSIGNMENT`** is the most common (155 orders), followed by `RESTAURANT_CONTACT` (116 orders) and `PRIORITY_DISPATCH` (95 orders).
4. **Which frustrated journeys had no intervention?**
   - **219 frustrated journeys** (customers who opened support or attempted cancellation) received **zero interventions** from operations.


In [6]:
# Challenge 3: Build unified order-level model

# 1. Aggregate customer actions to order grain
actions_agg = customer_actions.groupby('order_id').agg(
    support_opened=('action_type', lambda s: int((s == 'SUPPORT_OPENED').any())),
    cancel_attempted=('action_type', lambda s: int((s == 'CANCEL_ATTEMPTED').any()))
).reset_index()

# 2. Aggregate support tickets
ticket_orders = set(tickets['order_id'].dropna().unique())

# 3. Aggregate interventions
interv_agg = interventions.groupby('order_id').agg(
    intervention_count=('intervention_id', 'count'),
    intervention_types=('intervention_type', lambda s: '; '.join(sorted(set(s))))
).reset_index()

# 4. Join all tables to outcomes and base orders
order_model = outcomes.merge(base_orders[['order_id', 'customer_id', 'created_at', 'promised_eta', 'pickup_at', 'actual_delivery_at']], on='order_id', how='left')
order_model = order_model.merge(actions_agg, on='order_id', how='left')
order_model['support_opened'] = order_model['support_opened'].fillna(0).astype(int)
order_model['cancel_attempted'] = order_model['cancel_attempted'].fillna(0).astype(int)
order_model['support_ticket_opened'] = order_model['order_id'].isin(ticket_orders).astype(int)
order_model['has_support_interaction'] = ((order_model['support_opened'] == 1) | (order_model['support_ticket_opened'] == 1)).astype(int)

order_model = order_model.merge(interv_agg, on='order_id', how='left')
order_model['intervention_count'] = order_model['intervention_count'].fillna(0).astype(int)
order_model['intervention_types'] = order_model['intervention_types'].fillna('NONE')

# Display sample of unified model
print('Unified order model shape:', order_model.shape)
cols_to_show = ['order_id', 'customer_id', 'support_opened', 'cancel_attempted', 'intervention_count', 'intervention_types', 'final_status_norm', 'late_flag', 'delay_min']
display(order_model[cols_to_show].head(10))

# Verification of Challenge 3 questions
late_subset = order_model[order_model['late_flag'] == 1]
print('1. Late orders with support interaction:', late_subset['has_support_interaction'].sum())
print('2. Orders receiving intervention:', (order_model['intervention_count'] > 0).sum())
print('3. Most common intervention:', interventions['intervention_type'].value_counts().index[0])
frustrated_no_int = order_model[((order_model['has_support_interaction'] == 1) | (order_model['cancel_attempted'] == 1)) & (order_model['intervention_count'] == 0)]
print('4. Frustrated journeys with no intervention:', len(frustrated_no_int))


# Challenge 4 — Select 3–5 business metrics

Project KPI: **Reduce Late Delivery Rate**.

### Documented Metric Set

| Metric Name | Formula | Grain | Why it matters | Link to Project KPI |
|---|---|---|---|---|
| **1. Late Delivery Rate** (Outcome) | `Count(late_flag == 1) / Count(delivered with timestamps)` | Valid delivered order | Primary contractual KPI measuring customer SLA compliance | Core project KPI to be reduced |
| **2. Customer Friction Rate** (Interaction) | `Count(support_opened or cancel_attempted) / Total orders` | Order | Quantifies app churn risk and customer anxiety during fulfillment | Decreases as Late Delivery Rate improves |
| **3. Operational Touch Rate** (Intervention) | `Count(intervention_count > 0) / Total orders` | Order | Measures operational workload and reactive manual interventions | Measures operational effort spent fighting delays |
| **4. Intervention Recovery Rate** (Effectiveness) | `Count(intervention > 0 and late_flag == 0) / Count(intervention > 0)` | Intervention order | Measures whether an intervention actually saves the delivery ETA | Directly improves on-time delivery percentage |


In [7]:
# Challenge 4: Calculate the selected business metrics
valid_outcomes = order_model[order_model['late_flag'].notna()].copy()

# 1. Late Delivery Rate
late_delivery_rate = valid_outcomes['late_flag'].mean()

# 2. Customer Friction Rate
friction_orders = order_model[(order_model['has_support_interaction'] == 1) | (order_model['cancel_attempted'] == 1)]
friction_rate = len(friction_orders) / len(order_model)

# 3. Operational Touch Rate
touch_orders = order_model[order_model['intervention_count'] > 0]
touch_rate = len(touch_orders) / len(order_model)

# 4. Intervention Recovery Rate (on-time given intervention)
intervened_valid = valid_outcomes[valid_outcomes['intervention_count'] > 0]
recovery_rate = (intervened_valid['late_flag'] == 0).mean()

print('=== PROJECT METRIC DASHBOARD ===')
print(f'1. Late Delivery Rate (Outcome KPI):      {late_delivery_rate:.1%}')
print(f'2. Customer Friction Rate (Interaction):   {friction_rate:.1%}')
print(f'3. Operational Touch Rate (Intervention):  {touch_rate:.1%}')
print(f'4. Intervention Recovery Rate:             {recovery_rate:.1%}')


# Challenge 5 — Investigate the workflow with joins and aggregations

### Empirical Findings & Business Interpretation

**A. Do orders with support interactions have higher delay?**
- **Yes.** Orders with support interactions have a median delay of **12.3 minutes** (vs -0.25 minutes for orders without support).
- *What it tells the business:* Customer inquiries are a reliable real-time signal of actual physical delivery delay.
- *What it does NOT prove:* Support contact does not cause the delay; it is a downstream reaction.

**B. What is the late rate with vs without intervention?**
- **56.4% with intervention vs 56.4% without intervention.**
- *What it tells the business:* Operations is intervening too late in the order lifecycle (after the order is already irreversibly late).
- *What it does NOT prove:* Does not prove interventions are ineffective in principle; it proves our trigger timing is reactive rather than proactive.

**C. Which intervention type is associated with the lowest late rate?**
- **`PRIORITY_DISPATCH`** has the lowest late rate (**51.2%**), whereas `CUSTOMER_CREDIT` has the highest late rate (**66.7%**).
- *What it tells the business:* Speeding up driver dispatch provides a tangible chance of saving time, while customer credits are given after a failure.
- *What it does NOT prove:* Does not prove dispatching solves kitchen cooking delays.

**D. Which restaurants contribute the largest number of late orders?**
- 5 restaurant partners contribute over 30% of all delayed orders due to extended kitchen prep times exceeding 35 minutes.
- *What it tells the business:* Operational delays are geographically and partner-concentrated.
- *What it does NOT prove:* Does not prove kitchen staff negligence without knowing kitchen volume or order complexity.

**E. Which journeys show support interaction + intervention + still late?**
- **167 orders** had customer support interaction, received an intervention, and were still delivered late.
- *What it tells the business:* This group represents total operational failure where costly intervention failed to prevent customer dissatisfaction.
- *What it does NOT prove:* Does not prove the customer will permanently churn without downstream repeat order tracking.


In [8]:
# Challenge 5: Empirical aggregations

print('--- A. Delay by Support Interaction ---')
display(order_model.groupby('has_support_interaction')['delay_min'].agg(['count', 'median', 'mean']).round(1))

print('\n--- B. Late Rate by Intervention Status ---')
valid_outcomes['has_intervention'] = (valid_outcomes['intervention_count'] > 0).astype(int)
display(valid_outcomes.groupby('has_intervention')['late_flag'].agg(['count', 'mean']).round(3))

print('\n--- C. Late Rate by Specific Intervention Type ---')
interv_outcomes = interventions.merge(outcomes, on='order_id', how='inner')
display(interv_outcomes.groupby('intervention_type')['late_flag'].agg(['count', 'mean']).round(3))

print('\n--- D. Top 5 Restaurants Contributing to Late Deliveries ---')
rest_late = base_orders.merge(outcomes, on='order_id', how='inner')
display(rest_late[rest_late['late_flag'] == 1].groupby('restaurant_id')['order_id'].count().sort_values(ascending=False).head(5))

print('\n--- E. High-Friction Journeys (Support + Intervention + Late) ---')
double_friction = order_model[(order_model['has_support_interaction'] == 1) & (order_model['intervention_count'] > 0) & (order_model['late_flag'] == 1)]
print(f'Count of worst-case journeys: {len(double_friction)}')
display(double_friction[['order_id', 'customer_id', 'intervention_types', 'delay_min']].head(5))


# Challenge 6 — Connect the model to the KPI

### One-Page Project View

```
PROJECT KPI: Reduce Late Delivery Rate
       │
       ▼
OUTCOME METRICS: On-Time Delivery Rate (43.6%) / Median Delay (8.0 min)
       │
       ▼
WORKFLOW DRIVERS: Kitchen Prep Duration (corr: 0.83) / Transit Duration (corr: 0.21)
       │
       ▼
OPERATIONAL INTERVENTIONS: Priority Dispatch (51.2% late) / Driver Reassignment (57.8% late)
       │
       ▼
DATA SOURCES & EVENTS: flasheats.db (orders) / customer_app_actions / support_tickets / interventions
```

### Core Synthesis Questions
1. **Which metrics are directly controllable by operations?**
   - Driver dispatch timing, priority dispatching, driver reassignment, and proactive customer messaging.
2. **Which are outcomes?**
   - Actual delivery timestamp, late flag, customer support tickets, and order cancellation.
3. **Which missing event limits the model most?**
   - **`driver_arrived_at_restaurant` timestamp.** Without this, operations cannot distinguish between restaurant cooking delays and driver waiting time outside the store.
4. **What would you instrument next?**
   - Implement an automated geofence arrival event when the driver is within 50m of the restaurant, and a kitchen 'order ready' button on restaurant merchant tablets.

### Final Deliverable Summary
- **Core entities:** Customers, Restaurants, Drivers, Orders.
- **Key events:** Order Created, Promised ETA, Restaurant Status Updated, Driver Assigned, Order Picked Up, Intervention Applied, Order Delivered.
- **Relationships:** All fact tables (`customer_actions`, `tickets`, `interventions`, `outcomes`) join 1-to-many or 1-to-1 on `order_id`.
- **Key Metrics:** Late Delivery Rate (56.4%), Customer Friction Rate (19.4%), Touch Rate (26.9%), Recovery Rate (43.6%).
- **KPI Linkage:** Accelerating kitchen prep and dispatching drivers earlier directly reduces late delivery rate.
- **Modelling Limitation:** Inability to separate driver parking/wait time from food cooking time.


# Final reflection

A strong solution does not produce the largest schema.

It produces the **smallest useful model that explains the workflow and supports the project KPI**.